## How to use the sandbox with jupyter notebooks

Basically it's just this cell and you are up and running:

Some gotchas: 
1. Make sure you have an image with the chosen name
2. Make sure docker desktop is running

In [1]:
import importlib
import src.benchmark
importlib.reload(src.benchmark)
from src.benchmark import BenchmarkProblem, LightweightNotebook

sandbox_settings = {
    "image_name": "yarinamomo/kaggle_python_env", 
    "port": 8888, 
}

problem = BenchmarkProblem(sandbox_settings, "example/test/", "example/docker_mount/")

problem.setup(with_debugger=False)

🐳 Starting Sandbox Container...
✅ Sandbox Ready.


Now you can just juse the notebook like this:

In [2]:
problem.notebook.get_cells()

['# --- [CELL 0]: ---\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.neural_network import MLPClassifier\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Dense, Dropout\nfrom catboost import CatBoostClassifier\nfrom sklearn.metrics import accuracy_score',
 "# --- [CELL 1]: ---\ntrain_df = pd.read_csv('data/train_synthetic.csv')\ntest_df = pd.read_csv('data/test_synthetic.csv')\ngreeks_df = pd.read_csv('data/greeks_synthetic.csv')",
 '# --- [CELL 2]: ---\ntrain_df = pd.merge(train_df, greeks_df, on="Id")',
 '# --- [CELL 3]: ---\ntrain_df = train_df.drop("I

In [3]:
[r.display() for r in problem.notebook.run_all()]

---ERROR---: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- AB_0.0
- AF_0.0
- AH_0.0
- AM_0.0
- AR_0.0
- ...
Feature names seen at fit time, yet now missing:
- AB_0.11
- AB_0.12
- AB_0.16
- AB_0.18
- AB_0.19
- ...



[None, None, None, None, None, None]

In [5]:

print("=== Not Prints or returns===")
res = await problem.notebook.run_cell_async(1)
print(res.status)
print(res.result.display())
print("")
print("=== With prints / cell outputs ===")
res = await problem.notebook.run_cell_async(3)
print(res.status)
print(res.result.display())

=== Not Prints or returns===
ExecutionStatus.COMPLETED
None

=== With prints / cell outputs ===
ExecutionStatus.COMPLETED
None


finally teardown:

In [6]:
problem.teardown()

🛑 Sandbox Destroyed.


For agent loops you probably want a dependency object and a wrapper around an agent that contains this benchmark problem and write the tool calls to accept your dependency object. An example slightly shortened out of my tools definition:

```python
from pydantic_ai import RunContext

@dataclass
class AgentDependencies:
    problem: BenchmarkProblem
    
    @property
    def notebook(self) -> LightweightNotebook:
        """Convenience accessor for the notebook."""
        return self.problem.notebook


async def run_cell(ctx: RunContext[AgentDependencies], cell_index: int, performs_inference: bool) -> str:
    """
    Execute a specific cell by its index and return the output.
    
    Args:
        cell_index: The 0-based index of the cell to run
        performance_inference: True if the cell performs inference (i.e. runs pm.sample, or pmd.debug) (for logging purposes)
    """
    notebook = ctx.deps.notebook

    print("🤖 Running cell", cell_index)
    
    try:
        start_time = time.time()
        if performs_inference:
            ctx.deps.inference_count_agent += 1
            print(f"    ~ [Info]: Inference agent run count is now {ctx.deps.inference_count_agent}")
        
        will_run, inference_context, _ = code_will_run_inference(notebook.get_cell(cell_index), ctx.deps.inference_context)
        if will_run:
            ctx.deps.inference_count_static += 1
            print(f"    ~ [Info]: Inference static analysis run count is now {ctx.deps.inference_count_static}")

        ctx.deps.inference_context = inference_context

        result = await notebook.run_cell_async(cell_index)

        output = result.result.llm_compatible() if result.result else "(No output)"

        if code_ran_inference(result.result):
            ctx.deps.inference_count_post += 1
            elapsed = time.time() - start_time
            ctx.deps.time_spent_in_inference += elapsed
            print(f"    ~ [Info]: Inference post run count is now {ctx.deps.inference_count_post}")

        print(f"    ~ [Output]: ", "\n".join([m.content if m.type == SandboxResultType.TEXT else "<Image not rendered>" for m in output]))
        return output
    except IndexError:
        return f"Error: Cell index {cell_index} is out of range. Notebook has {notebook.get_cell_count()} cells."
    except Exception as e:
        return f"Error running cell {cell_index}: {str(e)}"
```